## Basic testing of RNN, LSTM, and GRU ##

In [128]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [129]:
def prepare_sequence(seq, to_ix):
    idxs = [to_ix[w] for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

# What will happen here?
training_data = [
    # Tags are: DET - determiner; NN - noun; V - verb
    # For example, the word "The" is a determiner
    ("The dog ate the apple".split(), ["DET", "NN", "V", "DET", "NN"]),
    ("Everybody read that book".split(), ["NN", "V", "DET", "NN"]),
    ("Everybody does machine learning nowadays".split(), ["NN", "V", "NN", "NN", "ADV", ])
]
word_to_ix = {}
# For each words-list (sentence) and tags-list in each tuple of training_data
for sent, tags in training_data:
    for word in sent:
        if word not in word_to_ix:  # word has not been assigned an index yet
            word_to_ix[word] = len(word_to_ix)  # Assign each word with a unique index
            
print(word_to_ix)
tag_to_ix = {"DET": 0, "NN": 1, "V": 2, "ADV": 3}  # Assign each tag with a unique index

{'The': 0, 'dog': 1, 'ate': 2, 'the': 3, 'apple': 4, 'Everybody': 5, 'read': 6, 'that': 7, 'book': 8, 'does': 9, 'machine': 10, 'learning': 11, 'nowadays': 12}


In [130]:
EMBEDDING_DIM = 6
HIDDEN_DIM = 12
VOCAB_SIZE = len(word_to_ix)
NUM_CLASSES = len(tag_to_ix)

In [131]:
def train(model, optimizer, criterion, epochs):
    epoch_loss = []
    for epoch in range(epochs):  # again, normally you would NOT do 300 epochs, it is toy data
        final_loss = 0
        for sentence, tags in training_data:
            
            model.zero_grad()

            # get inputs and targets ready for the network!
            sentence_in = prepare_sequence(sentence, word_to_ix)
            targets = prepare_sequence(tags, tag_to_ix)

            # get the tag scores
            tag_scores = model(sentence_in)
            
            loss = criterion(tag_scores, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            final_loss += loss.item()
        epoch_loss.append(final_loss)
    
    return epoch_loss

In [132]:
def evaluate(model, test_sequence):
    with torch.no_grad():
        inputs = prepare_sequence(training_data[test_sequence][0], word_to_ix)
        tag_scores = model(inputs)
        
        outputs = []
        
        print(tag_to_ix)
        print(training_data[test_sequence][0])
        print(training_data[test_sequence][1])
        
        for tag_score in tag_scores:
            outputs.append(tag_score.topk(1).indices.item())
            
        print(outputs)
        print("--------------")

### Recurrent Neural Networks (RNN) ###

In [133]:
class RNNTagger(nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(RNNTagger, self).__init__()
        self.hidden_dim = hidden_dim

        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # The RNN takes word embeddings as inputs, and outputs hidden states and output
        self.rnn = nn.RNN(embedding_dim, hidden_dim)

        # The linear layer that maps from hidden state space to tag space
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        
        embeds = self.word_embeddings(sentence)
        rnn_out, _ = self.rnn(embeds.view(len(sentence), 1, -1)) #The module is expecting [sentence_length, batch_size, embedding_dim]
        
        # in this case, rnn_out.view(len(sentence), -1) is the same as doing what function?
        tag_space = self.hidden2tag(rnn_out.view(len(sentence), -1))
        
        tag_scores = F.log_softmax(tag_space, dim=1)
        return tag_scores

In [134]:
model = RNNTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_ix), len(tag_to_ix))
loss_function = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)
rnn_losses = train(model, optimizer, loss_function, 100)
print(rnn_losses)
evaluate(model, 0)
evaluate(model, 1)
evaluate(model, 2)

[4.163668751716614, 3.7677351236343384, 3.442226529121399, 3.163565158843994, 2.9169872403144836, 2.691340148448944, 2.4781241416931152, 2.2720401883125305, 2.07133412361145, 1.8775027394294739, 1.6942272782325745, 1.5256077349185944, 1.3744003176689148, 1.2411887049674988, 1.1247119307518005, 1.0227667689323425, 0.9330200701951981, 0.8534512519836426, 0.7824799865484238, 0.7189213633537292, 0.6618818491697311, 0.6106566935777664, 0.5646542385220528, 0.5233490616083145, 0.48625892400741577, 0.45293690264225006, 0.42296984791755676, 0.3959801718592644, 0.3716265335679054, 0.3496037498116493, 0.32964111864566803, 0.31150053441524506, 0.29497307538986206, 0.27987609431147575, 0.2660503685474396, 0.2533563822507858, 0.24167272076010704, 0.23089348524808884, 0.22092554718255997, 0.21168730407953262, 0.2031068503856659, 0.1951214149594307, 0.18767501786351204, 0.18071839585900307, 0.17420747131109238, 0.16810322552919388, 0.16237123310565948, 0.1569800116121769, 0.15190192498266697, 0.147111

### Long Short-Term Memory (LSTM) ###

In [135]:
class LSTMTagger(nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTagger, self).__init__()
        self.hidden_dim = hidden_dim

        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # The LSTM takes word embeddings as inputs, and outputs hidden states
        # with dimensionality hidden_dim.
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)

        # The linear layer that maps from hidden state space to tag space
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        tag_scores = F.log_softmax(tag_space, dim=1)
        return tag_scores

In [136]:
model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_ix), len(tag_to_ix))
loss_function = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)
lstm_losses = train(model, optimizer, loss_function, 100)
print(lstm_losses)
evaluate(model, 0)
evaluate(model, 1)
evaluate(model, 2)

[4.007662057876587, 3.9260975122451782, 3.857724666595459, 3.80019474029541, 3.7515259981155396, 3.710062026977539, 3.674430251121521, 3.6435028314590454, 3.6163601875305176, 3.592254638671875, 3.5705827474594116, 3.550857424736023, 3.532686948776245, 3.5157558917999268, 3.499809741973877, 3.48464298248291, 3.4700887203216553, 3.456012010574341, 3.4423011541366577, 3.42886483669281, 3.415627360343933, 3.4025256633758545, 3.389506220817566, 3.376525044441223, 3.363543748855591, 3.3505300283432007, 3.3374555110931396, 3.324297070503235, 3.3110333681106567, 3.2976468801498413, 3.284122347831726, 3.2704458236694336, 3.2566057443618774, 3.2425923347473145, 3.2283960580825806, 3.2140095233917236, 3.1994258165359497, 3.1846383810043335, 3.1696414947509766, 3.1544305086135864, 3.138999819755554, 3.123345136642456, 3.107461452484131, 3.091343641281128, 3.0749871134757996, 3.058385729789734, 3.041534423828125, 3.024426519870758, 3.007054924964905, 2.9894123673439026, 2.971490442752838, 2.9532802

## Replace LSTM and RNN with GRU ##

Implement a network with nn.GRU, and compare with the other networks through loss and perplexity. If wanted, you can extend this toy example with more sentences or vary the task for testing the networks and observing the differences.

In [137]:
class GRUTagger(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(GRUTagger, self).__init__()
        self.hidden_dim = hidden_dim

        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # GRU layer
        self.gru = nn.GRU(embedding_dim, hidden_dim)

        # Linear output layer
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        gru_out, _ = self.gru(embeds.view(len(sentence), 1, -1))
        tag_space = self.hidden2tag(gru_out.view(len(sentence), -1))
        tag_scores = F.log_softmax(tag_space, dim=1)
        return tag_scores

In [138]:
# Instantiate GRU model
gru_model = GRUTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_ix), len(tag_to_ix))
loss_function = nn.NLLLoss()
optimizer = optim.SGD(gru_model.parameters(), lr=0.1)

# Train GRU model
gru_losses = train(gru_model, optimizer, loss_function, 100)
print("GRU Losses:", gru_losses)

# Evaluate GRU
evaluate(gru_model, 0)
evaluate(gru_model, 1)
evaluate(gru_model, 2)

GRU Losses: [4.1644957065582275, 4.004307866096497, 3.8780412673950195, 3.7784096002578735, 3.6993587017059326, 3.635886788368225, 3.5839492082595825, 3.5403677225112915, 3.502722144126892, 3.469220995903015, 3.438569664955139, 3.4098511934280396, 3.382423162460327, 3.355841040611267, 3.3297955989837646, 3.30407178401947, 3.2785152196884155, 3.2530146837234497, 3.22748339176178, 3.2018518447875977, 3.1760591864585876, 3.1500505805015564, 3.1237727403640747, 3.097174048423767, 3.0702024698257446, 3.042805850505829, 3.014932870864868, 2.9865322709083557, 2.957555413246155, 2.927955746650696, 2.89769047498703, 2.866721212863922, 2.835015833377838, 2.8025485277175903, 2.769301474094391, 2.7352644205093384, 2.7004372477531433, 2.6648297905921936, 2.628461480140686, 2.591362774372101, 2.5535759329795837, 2.5151522755622864, 2.4761549830436707, 2.4366554021835327, 2.39673388004303, 2.3564772605895996, 2.315977692604065, 2.2753294110298157, 2.2346279621124268, 2.193966805934906, 2.153435707092

# Compare losses

In [139]:
import math

def perplexity(loss):
    return math.exp(loss)

print("RNN Perplexity:", perplexity(rnn_losses[-1]))
print("LSTM Perplexity:", perplexity(lstm_losses[-1]))
print("GRU Perplexity:", perplexity(gru_losses[-1]))


RNN Perplexity: 1.0550982269359004
LSTM Perplexity: 5.330867318801387
GRU Perplexity: 1.8514099243044062
